# Data Cleaning e Arricchimento del Corpus Normativo UE

Questo notebook esegue il preprocessing del corpus normativo grezzo scaricato da EurLex e produce i file puliti e arricchiti che costituiscono l'input per tutti i notebook di analisi successivi.

## Dipendenze

- **Input** (da `data/raw/`):
  - `nodes.csv` — 88.130 atti normativi UE con metadati
  - `edges.csv` — 219.302 relazioni di citazione
  - `eurovoc_concept.csv` — 7.613 concetti EuroVoc con gerarchia
  - `has_concept_edges.csv` — 299.026 associazioni atto → concetto EuroVoc

- **Output** (in `data/processed/`):
  - `nodes_light.csv` — 40.604 atti normativi rilevanti con metadati puliti
  - `edges_enriched.csv` — archi filtrati tra nodi rilevanti, arricchiti con anno
  - `has_concept_enriched.csv` — associazioni EuroVoc filtrate per nodi rilevanti

## Operazioni eseguite

1. Pulizia e standardizzazione dei CELEX ID
2. Estrazione robusta dell'anno con fallback multipli
3. Classificazione del tipo di atto giuridico
4. Filtro per atti normativi rilevanti (esclusione atti preparatori, nazionali, ecc.)
5. Assegnazione di ere storiche per analisi temporale
6. Esportazione dei file processati

## 0. Setup

In [1]:
import pandas as pd
import re
import os
from datetime import datetime

# Percorsi dati
raw_path  = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

print(f"Input:  {os.path.abspath(raw_path)}")
print(f"Output: {os.path.abspath(proc_path)}")

Input:  c:\Users\claud\Documents\GitHub\eu-law-network-viz\data\raw
Output: c:\Users\claud\Documents\GitHub\eu-law-network-viz\data\processed


## 1. Caricamento Dati

In [2]:
print("Caricamento dati...")
nodes       = pd.read_csv(os.path.join(raw_path, 'nodes.csv'))
edges       = pd.read_csv(os.path.join(raw_path, 'edges.csv'))
eurovoc     = pd.read_csv(os.path.join(raw_path, 'eurovoc_concept.csv'))
has_concept = pd.read_csv(os.path.join(raw_path, 'has_concept_edges.csv'))

print(f"  Nodes:            {len(nodes):>7,}")
print(f"  Edges:            {len(edges):>7,}")
print(f"  EuroVoc concepts: {len(eurovoc):>7,}")
print(f"  Has-concept:      {len(has_concept):>7,}")

Caricamento dati...
  Nodes:             88,130
  Edges:            219,302
  EuroVoc concepts:   7,613
  Has-concept:      299,026


## 2. Pulizia CELEX ID

I CELEX ID nel dataset grezzo presentano due problemi:

1. **Caratteri non standard**: spazi, trattini, parentesi che rendono i join inconsistenti
2. **Corrigendum**: suffissi del tipo `R01`, `R02` che indicano rettifiche formali dello stesso atto, non atti distinti. Vengono normalizzati all'atto principale (es. `32022L2557R(01)` → `32022L2557`)

La pulizia è puramente formale e non altera il contenuto informativo dei metadati.

In [3]:
def clean_celex(celex):
    """Pulisce e standardizza i CELEX ID.
    
    Operazioni:
    - Rimuove caratteri non alfanumerici (spazi, trattini, parentesi)
    - Normalizza i corrigendum RNN → atto principale
    """
    if pd.isna(celex) or celex == '':
        return None
    celex = str(celex).strip()
    celex = re.sub(r'[^0-9A-Za-z]', '', celex)
    # Corrigendum: suffisso R seguito da 2 cifre (es. R01, R02)
    celex = re.sub(r'R\d{2}$', '', celex)
    return celex if celex else None

nodes['celex_clean'] = nodes['celex_id'].apply(clean_celex)

n_valid   = nodes['celex_clean'].notna().sum()
n_missing = nodes['celex_clean'].isna().sum()
print(f"CELEX validi:   {n_valid:,} ({n_valid/len(nodes)*100:.1f}%)")
print(f"CELEX mancanti: {n_missing:,} ({n_missing/len(nodes)*100:.1f}%)")

CELEX validi:   86,357 (98.0%)
CELEX mancanti: 1,773 (2.0%)


## 3. Estrazione Anno

La colonna `year` del dataset grezzo ha una copertura del ~39%. Per aumentarla, implementiamo una logica di estrazione con fallback a più livelli:

1. **Pattern CELEX — Trattati** (`1YYYY...`): l'anno è nei 4 digit dopo il settore 1
2. **Pattern CELEX — Atti moderni** (`[3-9]YYYY...`): l'anno è nei 4 digit dopo il settore
3. **Pattern generico**: prima sequenza di 4 digit plausibile come anno UE (1951–oggi)
4. **Fallback URL**: anno estratto dall'URL del documento nel formato `/YYYY/`

Tutti i pattern includono una validazione `1951 ≤ anno ≤ anno corrente` per evitare falsi positivi.

In [4]:
def extract_year(celex, row=None):
    """Estrae l'anno dall'ID CELEX con fallback multipli."""
    if pd.isna(celex) or celex == '':
        return None

    celex = str(celex)
    current_year = datetime.now().year

    def valid_year(y):
        return 1951 <= y <= current_year

    # Pattern 1: Trattati (settore 1 + YYYY)
    m = re.search(r'^1(\d{4})', celex)
    if m:
        y = int(m.group(1))
        if valid_year(y):
            return y

    # Pattern 2: Atti moderni (settori 3-9 + YYYY)
    m = re.search(r'^[3-9](\d{4})', celex)
    if m:
        y = int(m.group(1))
        if valid_year(y):
            return y

    # Pattern 3: Prima sequenza di 4 digit plausibile
    m = re.search(r'(\d{4})', celex)
    if m:
        y = int(m.group(1))
        if valid_year(y):
            return y

    # Pattern 4: Fallback URL
    if row is not None and pd.notna(row.get('url')):
        m = re.search(r'/(\d{4})/', str(row['url']))
        if m:
            y = int(m.group(1))
            if valid_year(y):
                return y

    return None

print("Estrazione anni...")
nodes['year_extracted'] = nodes.apply(
    lambda row: extract_year(row['celex_clean'], row), axis=1
)

# Combina: anno originale ha priorità, fallback sull'estratto
nodes['year_final'] = nodes['year'].fillna(nodes['year_extracted'])
nodes['decade']     = (nodes['year_final'] // 10 * 10).astype('Int64')

print(f"Anno originale presente: {nodes['year'].notna().sum():,} / {len(nodes):,} ({nodes['year'].notna().sum()/len(nodes)*100:.1f}%)")
print(f"Anno estratto presente:  {nodes['year_extracted'].notna().sum():,} / {len(nodes):,} ({nodes['year_extracted'].notna().sum()/len(nodes)*100:.1f}%)")
print(f"Anno finale presente:    {nodes['year_final'].notna().sum():,} / {len(nodes):,} ({nodes['year_final'].notna().sum()/len(nodes)*100:.1f}%)")

Estrazione anni...
Anno originale presente: 34,384 / 88,130 (39.0%)
Anno estratto presente:  84,892 / 88,130 (96.3%)
Anno finale presente:    84,897 / 88,130 (96.3%)


## 4. Classificazione del Tipo di Atto

Il tipo di atto viene classificato in base a due fonti, in ordine di priorità:

1. **Metadato `resource_legal_type`** del dataset originale (quando disponibile e non generico)
2. **Struttura del CELEX ID** secondo la codifica ufficiale EUR-Lex:
   - Settore `1` → Trattati
   - Settore `3` + tipo `R` → Regolamento, `L` → Direttiva, `D` → Decisione
   - Settore `5` → Atti preparatori
   - Settore `6` → Giurisprudenza
   - ecc.

**Nota tecnica**: il tipo è ricavato dalla lettera alfabetica che segue i 5 digit `[settore][anno]` nel CELEX, non da una semplice ricerca di caratteri nella stringa. Questo evita falsi positivi (es. classificare come Regulation un atto il cui CELEX contiene `R` in altra posizione).

In [5]:
def classify_legal_type(celex, resource_type):
    """Classifica il tipo di atto giuridico.
    
    Priorità:
    1. Metadato resource_legal_type (se informativo)
    2. Struttura CELEX ID (settore + tipo)
    """
    # Priorità 1: metadato esplicito (escludi valori generici)
    if pd.notna(resource_type) and resource_type not in ('work', 'Other', ''):
        return resource_type

    if pd.isna(celex):
        return 'Unknown'

    celex = str(celex)

    # Settore 1: Trattati
    if celex.startswith('1'):
        return 'Treaty'

    # Settori 3-9: estrai il tipo dalla lettera dopo [settore][YYYY]
    # es. 32019R0452 → settore=3, anno=2019, tipo=R → Regulation
    m = re.search(r'^[3-9]\d{4}([A-Z]+)', celex)
    if m:
        tipo = m.group(1)
        if   celex.startswith('3'):
            if   tipo.startswith('R'): return 'Regulation'
            elif tipo.startswith('L'): return 'Directive'
            elif tipo.startswith('D'): return 'Decision'
            else:                      return 'Legislative_Act'
        elif celex.startswith('4'):    return 'Complementary_Act'
        elif celex.startswith('5'):    return 'Preparatory_Act'
        elif celex.startswith('6'):    return 'Case_Law'
        elif celex.startswith('7'):    return 'National_Act'
        elif celex.startswith('8'):    return 'Report'
        elif celex.startswith('9'):    return 'Other_Act'

    return 'Unknown'

print("Classificazione tipi di atto...")
nodes['legal_type_class'] = nodes.apply(
    lambda row: classify_legal_type(row['celex_clean'], row['resource_legal_type']),
    axis=1
)

print("\nDistribuzione per tipo (tutti i nodi):")
print(nodes['legal_type_class'].value_counts().head(20).to_string())

Classificazione tipi di atto...

Distribuzione per tipo (tutti i nodi):
legal_type_class
Regulation           17152
Decision             12448
Preparatory_Act      12278
R                    11965
D                     9740
M                     6628
Unknown               3487
Directive             3275
Legislative_Act       2770
Treaty                2754
Case_Law              2205
L                     1608
Y                      450
Complementary_Act      320
H                      293
A                      201
Q                      107
O                       92
E                       69
B                       69


## 5. Filtro per Atti Normativi Rilevanti

Per l'analisi della struttura normativa vengono mantenuti solo gli atti che hanno una funzione normativa diretta:

| Tipo incluso | Motivazione |
|---|---|
| `Regulation` | Atto normativo direttamente applicabile |
| `Directive` | Atto normativo vincolante sul risultato |
| `Decision` | Atto vincolante per i destinatari |
| `Treaty` | Diritto primario, base giuridica fondamentale |
| `Legislative_Act` | Atti legislativi non classificabili nelle categorie precedenti |
| `Case_Law` | Giurisprudenza CGUE, fonte interpretativa vincolante |

Sono esclusi: atti preparatori (proposte, pareri), atti nazionali, relazioni, atti complementari. Questi introducono rumore nell'analisi della struttura normativa UE senza contribuire alle relazioni gerarchiche di interesse.

In [6]:
RELEVANT_TYPES = [
    'Regulation',
    'Directive',
    'Decision',
    'Treaty',
    'Legislative_Act',
    'Case_Law',
]

nodes_relevant = nodes[nodes['legal_type_class'].isin(RELEVANT_TYPES)].copy()

print(f"Nodi totali:    {len(nodes):>7,}")
print(f"Nodi rilevanti: {len(nodes_relevant):>7,} ({len(nodes_relevant)/len(nodes)*100:.1f}%)")
print(f"Nodi esclusi:   {len(nodes) - len(nodes_relevant):>7,}")
print()
print("Distribuzione per tipo (rilevanti):")
print(nodes_relevant['legal_type_class'].value_counts().to_string())

Nodi totali:     88,130
Nodi rilevanti:  40,604 (46.1%)
Nodi esclusi:    47,526

Distribuzione per tipo (rilevanti):
legal_type_class
Regulation         17152
Decision           12448
Directive           3275
Legislative_Act     2770
Treaty              2754
Case_Law            2205


## 6. Assegnazione Ere Storiche

Gli atti vengono raggruppati in **ere politiche** corrispondenti ai principali trattati e momenti di discontinuità istituzionale dell'UE. Questa colonna è utile per l'analisi temporale dell'evoluzione della struttura normativa.

| Era | Anni | Evento di riferimento |
|---|---|---|
| Pre-Treaty of Rome | < 1958 | Prima delle Comunità europee |
| Pre-Single European Act | 1958–1986 | Mercato comune |
| Pre-Maastricht | 1987–1992 | Atto Unico Europeo |
| 1990s | 1993–1999 | Trattato di Maastricht |
| Early 2000s | 2000–2004 | Allargamento UE |
| Late 2000s | 2005–2009 | Trattato di Lisbona |
| Early 2010s | 2010–2014 | Crisi finanziaria |
| Late 2010s | 2015–2019 | Brexit, agenda digitale |
| 2020s | 2020– | Pandemia, Green Deal, AI Act |

In [7]:
def assign_era(year):
    """Assegna un'era storica in base all'anno."""
    if pd.isna(year):
        return 'Unknown'
    if year < 1958:  return 'Pre-Treaty of Rome'
    if year < 1987:  return 'Pre-Single European Act'
    if year < 1993:  return 'Pre-Maastricht'
    if year < 2000:  return '1990s'
    if year < 2005:  return 'Early 2000s'
    if year < 2010:  return 'Late 2000s'
    if year < 2015:  return 'Early 2010s'
    if year < 2020:  return 'Late 2010s'
    return '2020s'

nodes_relevant['era'] = nodes_relevant['year_final'].apply(assign_era)

print("Distribuzione per era:")
print(nodes_relevant['era'].value_counts().sort_index().to_string())

Distribuzione per era:
era
1990s                      4994
2020s                      6126
Early 2000s                4160
Early 2010s                6873
Late 2000s                 5010
Late 2010s                 6508
Pre-Maastricht             2518
Pre-Single European Act    3968
Pre-Treaty of Rome          429
Unknown                      18


## 7. Normalizzazione Tipo di Atto

Il dataset originale contiene talvolta codici brevi (es. `R`, `L`, `D`) accanto alle etichette estese. Questa cella li unifica in un'unica colonna `legal_type_normalized` con etichette consistenti.

In [8]:
# I tipi già classificati in legal_type_class sono già normalizzati.
# Questa colonna è un alias esplicito per chiarezza nei notebook successivi.
nodes_relevant['legal_type_normalized'] = nodes_relevant['legal_type_class']

# Verifica di consistenza: non ci devono essere valori fuori da RELEVANT_TYPES
unexpected = nodes_relevant[
    ~nodes_relevant['legal_type_normalized'].isin(RELEVANT_TYPES)
]
if len(unexpected) > 0:
    print(f"ATTENZIONE: {len(unexpected)} nodi con tipo inatteso:")
    print(unexpected['legal_type_normalized'].value_counts())
else:
    print("Verifica superata: tutti i nodi hanno un tipo valido.")

Verifica superata: tutti i nodi hanno un tipo valido.


## 8. Esportazione

### 8a. `nodes_light.csv`

Versione alleggerita del dataset nodi con sole le colonne necessarie per l'analisi. Riduce il carico in memoria e in Gephi.

In [9]:
NODES_LIGHT_COLS = [
    ':ID',
    'celex_id',
    'celex_clean',
    'work_title',
    'year_final',
    'legal_type_normalized',
    'era',
    'decade',
    'eurovoc_concepts:STRING[]',
    'domains:STRING[]',
    'subdomains:STRING[]',
    'url',
    'sector',
    'sector_label',
    'resource_legal_type',
    'legal_type_class',
]

# Seleziona solo le colonne disponibili (robustezza a variazioni del dataset)
available_cols = [c for c in NODES_LIGHT_COLS if c in nodes_relevant.columns]
missing_cols   = [c for c in NODES_LIGHT_COLS if c not in nodes_relevant.columns]
if missing_cols:
    print(f"Colonne non trovate (ignorate): {missing_cols}")

nodes_light = nodes_relevant[available_cols].copy()
nodes_light.to_csv(os.path.join(proc_path, 'nodes_light.csv'), index=False)
print(f"nodes_light.csv salvato: {len(nodes_light):,} righe, {len(nodes_light.columns)} colonne")

nodes_light.csv salvato: 40,604 righe, 16 colonne


### 8b. `edges_enriched.csv`

Archi filtrati per mantenere solo le citazioni tra nodi rilevanti. Aggiunge le colonne `start_year` e `end_year` per le analisi temporali.

In [10]:
relevant_node_ids    = set(nodes_relevant[':ID'].unique())
relevant_celex_clean = set(nodes_relevant['celex_clean'].dropna())
celex_to_raw_id      = dict(zip(nodes_relevant['celex_clean'], nodes_relevant[':ID']))

# Filtra: mantieni archi dove entrambi i nodi sono rilevanti
# Match su :ID grezzo OPPURE su celex_clean (copertura doppio formato)
edges_filtered = edges[
    (edges[':START_ID'].isin(relevant_node_ids) | edges[':START_ID'].isin(relevant_celex_clean)) &
    (edges[':END_ID'].isin(relevant_node_ids)   | edges[':END_ID'].isin(relevant_celex_clean))
].copy()

# Normalizza gli ID al formato grezzo (:ID) usato in nodes_light
edges_filtered[':START_ID'] = edges_filtered[':START_ID'].apply(
    lambda x: celex_to_raw_id.get(x, x)
)
edges_filtered[':END_ID'] = edges_filtered[':END_ID'].apply(
    lambda x: celex_to_raw_id.get(x, x)
)

# Arricchisci con anno di origine e destinazione
year_map = nodes_relevant.set_index(':ID')['year_final'].to_dict()
edges_filtered['start_year'] = edges_filtered[':START_ID'].map(year_map)
edges_filtered['end_year']   = edges_filtered[':END_ID'].map(year_map)

edges_filtered.to_csv(os.path.join(proc_path, 'edges_enriched.csv'), index=False)

print(f"Archi totali nel raw:      {len(edges):>7,}")
print(f"Archi tra nodi rilevanti:  {len(edges_filtered):>7,} ({len(edges_filtered)/len(edges)*100:.1f}%)")
print()
print("Per tipo di relazione:")
print(edges_filtered[':TYPE'].value_counts().to_string())

Archi totali nel raw:      219,302
Archi tra nodi rilevanti:   94,243 (43.0%)

Per tipo di relazione:
:TYPE
CITES                          49739
BASED_ON                       24539
AMENDS                          7330
CORRECTS                        4471
REPEALS                         3347
IMPLICITLY_REPEALS              2773
DOES_REPLACEMENT                 508
DEROGATES                        319
COMPLETES                        304
EXTENDS_VALIDITY                 268
DOES_INSERTION                   178
DOES_DELETION                    143
REPLACES                         103
IMPLEMENTS                        60
RELATED_TO                        48
DOES_REPEAL                       46
EXTENDS_APPLICATION               38
ADDS_TO                            9
ADOPTS                             6
SUSPENDS                           6
INTERPRETES_AUTHORITATIVELY        2
DEFERS_APPLICATION                 2
PROPOSES_TO_AMEND                  1
RELATED_QUESTION_TO                1
PART

### 8c. `has_concept_enriched.csv`

Associazioni EuroVoc filtrate per mantenere solo quelle relative ai nodi rilevanti.

In [11]:
has_concept_filtered = has_concept[
    has_concept[':START_ID'].isin(relevant_node_ids)
].copy()

has_concept_filtered.to_csv(os.path.join(proc_path, 'has_concept_enriched.csv'), index=False)
print(f"has_concept_enriched.csv salvato: {len(has_concept_filtered):,} righe")

has_concept_enriched.csv salvato: 137,980 righe


## 9. Verifica Qualità e Riepilogo Finale

In [12]:
print("=" * 55)
print("RIEPILOGO DATA CLEANING")
print("=" * 55)
print()
print("CORPUS GREZZO")
print(f"  Nodi totali:          {len(nodes):>7,}")
print(f"  Archi totali:         {len(edges):>7,}")
print()
print("CORPUS PROCESSATO")
print(f"  Nodi rilevanti:       {len(nodes_light):>7,} ({len(nodes_light)/len(nodes)*100:.1f}%)")
print(f"  Archi rilevanti:      {len(edges_filtered):>7,} ({len(edges_filtered)/len(edges)*100:.1f}%)")
print(f"  Has-concept filtrati: {len(has_concept_filtered):>7,}")
print()
print("QUALITÀ DEI DATI")
print(f"  Nodi con CELEX valido:   {nodes_light['celex_clean'].notna().sum():,} / {len(nodes_light):,} ({nodes_light['celex_clean'].notna().sum()/len(nodes_light)*100:.1f}%)")
print(f"  Nodi con anno:           {nodes_light['year_final'].notna().sum():,} / {len(nodes_light):,} ({nodes_light['year_final'].notna().sum()/len(nodes_light)*100:.1f}%)")
print(f"  Nodi con concetti EV:    {nodes_light['eurovoc_concepts:STRING[]'].notna().sum():,} / {len(nodes_light):,} ({nodes_light['eurovoc_concepts:STRING[]'].notna().sum()/len(nodes_light)*100:.1f}%)")
if 'work_title' in nodes_light.columns:
    print(f"  Nodi con titolo:         {nodes_light['work_title'].notna().sum():,} / {len(nodes_light):,} ({nodes_light['work_title'].notna().sum()/len(nodes_light)*100:.1f}%)")
print()
print("FILE SALVATI IN data/processed/")
print(f"  nodes_light.csv           ({len(nodes_light):,} righe)")
print(f"  edges_enriched.csv        ({len(edges_filtered):,} righe)")
print(f"  has_concept_enriched.csv  ({len(has_concept_filtered):,} righe)")
print()
print("Copertura temporale:")
if nodes_light['year_final'].notna().any():
    print(f"  {nodes_light['year_final'].min():.0f} – {nodes_light['year_final'].max():.0f}")

RIEPILOGO DATA CLEANING

CORPUS GREZZO
  Nodi totali:           88,130
  Archi totali:         219,302

CORPUS PROCESSATO
  Nodi rilevanti:        40,604 (46.1%)
  Archi rilevanti:       94,243 (43.0%)
  Has-concept filtrati: 137,980

QUALITÀ DEI DATI
  Nodi con CELEX valido:   40,604 / 40,604 (100.0%)
  Nodi con anno:           40,586 / 40,604 (100.0%)
  Nodi con concetti EV:    20,961 / 40,604 (51.6%)
  Nodi con titolo:         0 / 40,604 (0.0%)

FILE SALVATI IN data/processed/
  nodes_light.csv           (40,604 righe)
  edges_enriched.csv        (94,243 righe)
  has_concept_enriched.csv  (137,980 righe)

Copertura temporale:
  1951 – 2025
